# Endometrial Cancer (EC) Cohort Characteristics

Computes baseline characteristics (Table 2), BMI and ancestry composition, and the incident/prevalent case split for the final analysis cohort produced by `EC_Cohort_Definition.ipynb`.


## Setup


In [ ]:
import os
import pandas as pd
import numpy as np
import pandas_gbq
import subprocess
from datetime import datetime


In [ ]:
%run /home/jupyter/Var_set_2.ipynb  # Var_set_2.ipynb is provided automatically by the AoU Workbench (unique per user/project)
project_id = os.environ.get('GOOGLE_PROJECT', '').strip().strip("'\"")
CDR = os.environ['WORKSPACE_CDR']
print(project_id)

# dataset (from Var_set_2.ipynb) and CDR (WORKSPACE_CDR) are both used, showing they're the same
assert dataset == CDR, (
    f"'dataset' ({dataset!r}) and 'CDR' ({CDR!r}) do not match -- queries in this "
    f"notebook use both names and assume they refer to the same CDR release."
)
print(f"Confirmed: dataset and CDR both resolve to {dataset}")


## Baseline characteristics table


In [ ]:
# Loading the final cohort produced by EC_Cohort_Definition.ipynb
final_cohort = pd.read_csv(f"{bucket_output}/02_ec_final_cohort_excluded.csv")
print(f"Final cohort: {len(final_cohort):,}")
print(f"  Cases: {(final_cohort['EC_case']==1).sum():,}")
print(f"  Controls: {(final_cohort['EC_case']==0).sum():,}")


In [ ]:
# Merging birth dates and computing current age and age at diagnosis
query_birth = f"""
SELECT person_id, birth_datetime
FROM `{dataset}.person`
"""
birth_dates = pandas_gbq.read_gbq(query_birth, project_id=project_id, dialect="standard")

final_cohort = final_cohort.merge(birth_dates, on='person_id', how='left')
final_cohort['birth_datetime'] = pd.to_datetime(final_cohort['birth_datetime']).dt.tz_localize(None)

# Fixed reference date for control age, so this figure is reproducible regardless of when the notebook is re-run (matches the date in the dissertation's Table 1 footnote)
current_date = pd.to_datetime('2026-10-22')
final_cohort['age_current'] = (current_date - final_cohort['birth_datetime']).dt.days / 365.25

final_cohort['first_diagnosis_date'] = pd.to_datetime(final_cohort['first_diagnosis_date'], errors='coerce')
final_cohort['age_at_diagnosis'] = (final_cohort['first_diagnosis_date'] - final_cohort['birth_datetime']).dt.days / 365.25

final_cohort['age_display'] = np.where(final_cohort['EC_case']==1, final_cohort['age_at_diagnosis'], final_cohort['age_current'])

print(f"Missing birth date: {final_cohort['birth_datetime'].isna().sum():,}")
print(final_cohort.groupby('EC_case')['age_display'].describe())


In [ ]:
# Querying menopause status and merging onto the cohort
cohort_id_set = set(int(x) for x in final_cohort['person_id'])

query_menopause = f"""
SELECT DISTINCT
    m.person_id,
    m.observation_date,
    c.concept_name as menopause_status
FROM `{dataset}.observation` m
JOIN `{dataset}.concept` c ON m.value_source_concept_id = c.concept_id
WHERE m.observation_source_concept_id = 1585784
"""
menopause_data = pandas_gbq.read_gbq(query_menopause, project_id=project_id, dialect="standard")
menopause_cohort = menopause_data[menopause_data['person_id'].astype(int).isin(cohort_id_set)].copy()

# The source query is DISTINCT on (person_id, observation_date, menopause_status), not on person_id alone as otherwise someone with multiple dated observations would fan out the merge below
# Keep only the most recent observation per person
n_before_dedup = len(menopause_cohort)
menopause_cohort = (
    menopause_cohort
    .sort_values('observation_date')
    .groupby('person_id', as_index=False)
    .last()
)
n_after_dedup = len(menopause_cohort)
if n_before_dedup != n_after_dedup:
    print(f"NOTE: {n_before_dedup - n_after_dedup:,} duplicate person-level menopause observations "
          f"collapsed to most-recent-per-person ({n_before_dedup:,} -> {n_after_dedup:,} rows).")
assert menopause_cohort['person_id'].is_unique, "menopause_cohort must be unique per person_id before merging"

n_before_merge = len(final_cohort)
final_cohort = final_cohort.merge(
    menopause_cohort[['person_id', 'menopause_status']], on='person_id', how='left'
)
assert len(final_cohort) == n_before_merge, "Merge changed row count -- menopause_cohort was not unique per person_id"

print(f"Coverage: {final_cohort['menopause_status'].notna().sum():,} / {len(final_cohort):,} "
      f"({final_cohort['menopause_status'].notna().sum()/len(final_cohort)*100:.1f}%)")
print(pd.crosstab(final_cohort['menopause_status'], final_cohort['EC_case'], normalize='columns') * 100)


In [ ]:
# Consent/enrolment date for everyone (concept 1586099)
all_ids = final_cohort['person_id'].astype(int).tolist()

def chunks(l, n):
    for i in range(0, len(l), n):
        yield l[i:i+n]

enrol_parts = []
for b in chunks(all_ids, 40000):
    s = ','.join(map(str, b))
    q = f"""
    SELECT person_id, MIN(observation_date) AS enroll_date
    FROM `{CDR}.observation`
    WHERE person_id IN ({s}) AND observation_source_concept_id = 1586099
    GROUP BY person_id
    """
    enrol_parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
enroll = pd.concat(enrol_parts, ignore_index=True)
enroll['enroll_date'] = pd.to_datetime(enroll['enroll_date'])
print(f"Consent date found: {enroll['enroll_date'].notna().sum():,} / {len(all_ids):,}")


In [ ]:
# All BMI measurements for everyone (concept 3038553, Body mass index)
bmi_parts = []
for b in chunks(all_ids, 40000):
    s = ','.join(map(str, b))
    q = f"""
    SELECT person_id, measurement_id, measurement_date, value_as_number AS bmi
    FROM `{CDR}.measurement`
    WHERE person_id IN ({s})
      AND measurement_concept_id = 3038553
      AND value_as_number IS NOT NULL
      AND value_as_number BETWEEN 10 AND 100
    """
    bmi_parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
bmi_all = pd.concat(bmi_parts, ignore_index=True)
bmi_all['measurement_date'] = pd.to_datetime(bmi_all['measurement_date'])
print(f"BMI records: {len(bmi_all):,} covering {bmi_all['person_id'].nunique():,} people")


In [ ]:
# Baseline BMI = measurement closest to consent date 
# Merging this onto the cohort
b2 = bmi_all.merge(enroll, on='person_id', how='inner')
b2['days_from_enroll'] = (b2['measurement_date'] - b2['enroll_date']).dt.days.abs()

# Deterministic tie-break when measurements are equally close to enrolment (prefer the earlier date, then the lower measurement_id)
# This avoids depending on random BigQuery row order (won't be stable across re-runs)
baseline_bmi = (b2.sort_values(['person_id', 'days_from_enroll', 'measurement_date', 'measurement_id'])
                   .groupby('person_id').first().reset_index()
                   [['person_id','bmi','measurement_date','days_from_enroll']]
                   .rename(columns={'bmi':'bmi_baseline','measurement_date':'bmi_baseline_date'}))
assert baseline_bmi['person_id'].is_unique, "Duplicate person_id after baseline BMI selection"

bmi_var = bmi_all.groupby('person_id')['bmi'].agg(
    bmi_sd='std', bmi_n_measurements='count', bmi_min='min', bmi_max='max'
).reset_index()

n_before_merge = len(final_cohort)
final_cohort = final_cohort.merge(baseline_bmi, on='person_id', how='left')
final_cohort = final_cohort.merge(bmi_var, on='person_id', how='left')
assert len(final_cohort) == n_before_merge, "BMI merge changed row count -- check for duplicate person_id upstream"

def bmi_category(b):
    if pd.isna(b): return np.nan
    elif b < 25: return '<25'
    elif b < 30: return '25-30'
    else: return '>=30'
final_cohort['bmi_category'] = final_cohort['bmi_baseline'].apply(bmi_category)

N = len(final_cohort)
print(f"Have a baseline BMI: {final_cohort['bmi_baseline'].notna().sum():,} / {N:,} "
      f"({100*final_cohort['bmi_baseline'].notna().mean():.1f}%)")
print(f"Median |days| between baseline BMI and consent: {final_cohort['days_from_enroll'].median():.0f}")
print(f"Median within-person BMI SD (>=2 measurements): "
      f"{final_cohort.loc[final_cohort['bmi_n_measurements']>=2,'bmi_sd'].median():.2f}")


In [ ]:
def summarize_continuous(df, col):
    return df.groupby('EC_case')[col].agg(['mean','std']).reset_index()

table1_rows = []
table1_rows.append(['N', f"{(final_cohort['EC_case']==0).sum():,}", f"{(final_cohort['EC_case']==1).sum():,}"])

age_summary = summarize_continuous(final_cohort, 'age_display')
table1_rows.append(['Age, mean (SD)',
    f"{age_summary.loc[0,'mean']:.1f} ({age_summary.loc[0,'std']:.1f})",
    f"{age_summary.loc[1,'mean']:.1f} ({age_summary.loc[1,'std']:.1f})"])

bmi_summary = summarize_continuous(final_cohort, 'bmi_baseline')
table1_rows.append(['BMI at baseline, mean (SD)',
    f"{bmi_summary.loc[0,'mean']:.1f} ({bmi_summary.loc[0,'std']:.1f})",
    f"{bmi_summary.loc[1,'mean']:.1f} ({bmi_summary.loc[1,'std']:.1f})"])

bmi_pct = pd.crosstab(final_cohort['bmi_category'], final_cohort['EC_case'], normalize='columns') * 100
for cat in ['<25', '25-30', '>=30']:
    table1_rows.append([f"BMI {cat}, %", f"{bmi_pct.loc[cat,0]:.1f}", f"{bmi_pct.loc[cat,1]:.1f}"])

cov = final_cohort.groupby('EC_case')['bmi_baseline'].apply(lambda s: 100*s.notna().mean())
table1_rows.append(['BMI at baseline available, %', f"{cov[0]:.1f}", f"{cov[1]:.1f}"])

gap_med = final_cohort.groupby('EC_case')['days_from_enroll'].median()
table1_rows.append(['BMI-to-consent gap, median days',
    f"{gap_med.get(0, float('nan')):.0f}", f"{gap_med.get(1, float('nan')):.0f}"])

sd_med = final_cohort.loc[final_cohort['bmi_n_measurements']>=2].groupby('EC_case')['bmi_sd'].median()
table1_rows.append(['Within-person BMI SD, median (>=2 msmts)',
    f"{sd_med.get(0, float('nan')):.2f}", f"{sd_med.get(1, float('nan')):.2f}"])

nmeas_med = final_cohort.groupby('EC_case')['bmi_n_measurements'].median()
table1_rows.append(['BMI measurements per person, median',
    f"{nmeas_med.get(0, float('nan')):.0f}", f"{nmeas_med.get(1, float('nan')):.0f}"])

meno_pct = pd.crosstab(final_cohort['menopause_status'], final_cohort['EC_case'], normalize='columns') * 100
if 'Menstrual Stopped: Yes None' in meno_pct.index:
    table1_rows.append(['Postmenopausal, %',
        f"{meno_pct.loc['Menstrual Stopped: Yes None',0]:.1f}",
        f"{meno_pct.loc['Menstrual Stopped: Yes None',1]:.1f}"])

anc_pct = pd.crosstab(final_cohort['ancestry_pred'], final_cohort['EC_case'], normalize='columns') * 100

# Coarsening small ancestry-group (eas/sas/mid) case percentages for AoU disclosure
SMALL_CELL_ANCESTRIES = {'eas', 'sas', 'mid'}

def coarsen_case_pct(anc, pct):
    if anc not in SMALL_CELL_ANCESTRIES:
        return f"{pct:.1f}"
    if pct < 1:
        return "<1"
    elif pct < 2:
        return "1-2"
    else:
        return f"{pct:.1f}"

for anc in ['eur', 'amr', 'afr', 'eas', 'sas', 'mid', 'oth']:
    if anc in anc_pct.index:
        ctrl_disp = f"{anc_pct.loc[anc,0]:.1f}"
        case_disp = coarsen_case_pct(anc, anc_pct.loc[anc,1])
        table1_rows.append([f"Ancestry: {anc}, %", ctrl_disp, case_disp])

table1_df = pd.DataFrame(table1_rows, columns=['Characteristic', 'Controls', 'Cases'])
print(table1_df.to_string(index=False))


## BMI and ancestry composition figure


In [ ]:
# Cohort composition by case status: BMI density (Panel A) and ancestry composition (Panel B).
# Reuses final_cohort / bmi_pct / anc_pct already computed above (so no new query or modelling)
# House style is centralised in dissertation_plot_style.py
%run /home/jupyter/dissertation_plot_style.py
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(5.8, 6.4))

# Panel A: BMI density by case status, within-group normalised
ax = axes[0]
bmi_all_vals = final_cohort['bmi_baseline'].dropna()
n_outside_1070 = int(((bmi_all_vals < 10) | (bmi_all_vals > 70)).sum())
pct_outside_1070 = 100 * n_outside_1070 / len(bmi_all_vals)
print(f"BMI outside [10,70] display range: {n_outside_1070} / {len(bmi_all_vals)} ({pct_outside_1070:.2f}%) -- "
      f"xlim(10,70) excludes a negligible tail, not a meaningful share of the distribution.")
for grp, label, color in [(0, 'Controls', GREY), (1, 'Cases', NAVY)]:
    sub = final_cohort.loc[final_cohort['EC_case'] == grp, 'bmi_baseline'].dropna()
    sns.kdeplot(sub, ax=ax, color=color, fill=True, alpha=0.25, linewidth=1.8, label=label)
ax.set_xlabel('BMI at baseline')
ax.set_ylabel('Density (within-group)')
ax.set_xlim(10, 70)
ax.set_title('A. Baseline BMI by case status', loc='left', fontsize=11, fontweight='bold')
ax.text(0.97, 0.90, f"Mean BMI\nCases: {bmi_summary.loc[1,'mean']:.1f}\nControls: {bmi_summary.loc[0,'mean']:.1f}", transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='#333333')
style_ax(ax)
ax.legend(frameon=False)

# Panel B: ancestry composition, paired horizontal bars
# Disclosure-safe subset -- East Asian, South Asian and Middle Eastern have case counts small enough that Table 1 coarsens their case percentages ('1-2%', '<1%')
# Plotting exact anc_pct values here would defeat that suppression, so this panel shows only the four groups large enough to report exactly. 
# The remaining three are reported, coarsened, in Table 1 only.
ax = axes[1]
anc_order = ['European', 'Admixed American', 'African', 'Other/unclassified']
anc_code = {'European':'eur','Admixed American':'amr','African':'afr','Other/unclassified':'oth'}
y_pos = np.arange(len(anc_order))[::-1]
case_pct = [anc_pct.loc[anc_code[a], 1] if anc_code[a] in anc_pct.index else 0 for a in anc_order]
ctrl_pct = [anc_pct.loc[anc_code[a], 0] if anc_code[a] in anc_pct.index else 0 for a in anc_order]

ax.barh(y_pos + 0.18, ctrl_pct, height=0.32, color=GREY, label='Controls')
ax.barh(y_pos - 0.18, case_pct, height=0.32, color=NAVY, label='Cases')
ax.set_yticks(y_pos)
ax.set_yticklabels(anc_order)
ax.set_xlabel('% of group')
ax.set_xlim(0, max(case_pct + ctrl_pct) * 1.22)
ax.set_title('B. Ancestry composition by case status', loc='left', fontsize=11, fontweight='bold')
eur_i = anc_order.index('European')
eur_y = y_pos[eur_i]
eur_case = case_pct[eur_i]
ax.text(eur_case + 1.5, eur_y - 0.18, f"{eur_case:.1f}%", va='center', ha='left', fontsize=9, color=NAVY)
ax.text(ctrl_pct[eur_i] + 1.5, eur_y + 0.18, f"{ctrl_pct[eur_i]:.1f}%", va='center', ha='left', fontsize=9, color='#666666')
style_ax(ax)
ax.legend(frameon=False)

plt.tight_layout()
savefig_both(fig, 'cohort_composition_bmi_ancestry', project_id, bucket_output)
plt.show()


In [ ]:
# Age at AoU enrolment, a common reference time for cases and controls, used as a sensitivity check
# Checking against age_display which isn't a common reference time (age-at-diagnosis for cases, fixed reference date for controls)
final_cohort = final_cohort.merge(enroll[['person_id', 'enroll_date']], on='person_id', how='left')
final_cohort['age_at_enrolment'] = (final_cohort['enroll_date'] - final_cohort['birth_datetime']).dt.days / 365.25

print(f"Missing enrolment date: {final_cohort['enroll_date'].isna().sum():,} / {len(final_cohort):,}")
print(final_cohort.groupby('EC_case')['age_at_enrolment'].describe())


## Incident vs prevalent classification


In [ ]:
# Incident vs prevalent (relative to consent date) and ancestry breakdown on the final cohort
case_ids = final_cohort.loc[final_cohort['EC_case']==1,'person_id'].astype(int).tolist()

parts = []
for b in chunks(case_ids, 40000):
    s = ','.join(map(str, b))
    q = f"""
    SELECT person_id, MIN(observation_date) AS consent_date
    FROM `{CDR}.observation`
    WHERE person_id IN ({s}) AND observation_source_concept_id = 1586099
    GROUP BY person_id
    """
    parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
consent = pd.concat(parts, ignore_index=True)
consent['consent_date'] = pd.to_datetime(consent['consent_date'])

dx = final_cohort.loc[final_cohort['EC_case']==1, ['person_id','first_diagnosis_date']].copy()
dx['first_diagnosis_date'] = pd.to_datetime(dx['first_diagnosis_date'])
m = dx.merge(consent, on='person_id', how='left')
m['status'] = np.select(
    [m['consent_date'].isna() | m['first_diagnosis_date'].isna(),
     m['first_diagnosis_date'] < m['consent_date']],
    ['UNKNOWN', 'PREVALENT'],
    default='INCIDENT'
)
n_unknown = (m['status'] == 'UNKNOWN').sum()
n_missing_dx = m['first_diagnosis_date'].isna().sum()
if n_unknown > 0:
    print(f"WARNING: {n_unknown} case(s) classified UNKNOWN (missing consent date and/or missing diagnosis date).")
    if n_missing_dx > 0:
        print(f"  Of these, {n_missing_dx} have a missing first_diagnosis_date specifically.")
assert m['first_diagnosis_date'].notna().all(), (
    f"{m['first_diagnosis_date'].isna().sum()} case(s) have no first_diagnosis_date at all -- "
    f"this should not happen for a final, cleaned cohort; investigate before proceeding."
)

print("=== Incident vs Prevalent ===")
print(m['status'].value_counts())
print(f"% incident: {100*(m['status']=='INCIDENT').mean():.1f}%")

# Merging this classification back onto final_cohort so it's carried in the saved output file
# Controls get NaN (never classified) which is expected
n_before_status_merge = len(final_cohort)
final_cohort = final_cohort.merge(
    m[['person_id', 'status']].rename(columns={'status': 'incident_status'}),
    on='person_id', how='left'
)
assert len(final_cohort) == n_before_status_merge, "incident_status merge changed row count -- check for duplicate person_id in m"

print("\n=== Cohort by genetic ancestry ===")
tab = final_cohort.groupby('ancestry_pred')['EC_case'].agg(
    controls=lambda s: int((s==0).sum()), cases=lambda s: int((s==1).sum()), total='count')
tab['case_pct_within_group'] = (100*tab['cases']/tab['total']).round(2)
tab['pct_of_all_cases'] = (100*tab['cases']/tab['cases'].sum()).round(1)
tab['pct_of_all_controls'] = (100*tab['controls']/tab['controls'].sum()).round(1)

# Reusing coarsen_case_pct/SMALL_CELL_ANCESTRIES defined in the Table 1 cell above
def coarsen_case_count(n):
    if n < 20:
        return '<20'
    lo = (n // 10) * 10 - 10
    hi = lo + 20
    return f"{lo}-{hi}"

# Building a display copy so the underlying numeric `tab` (used only for sorting/printing here) is untouched
# Nothing downstream reads `tab`'s small-group cells.
tab_display = tab.astype(object).copy()
for anc in SMALL_CELL_ANCESTRIES:
    if anc in tab_display.index:
        tab_display.loc[anc, 'cases'] = coarsen_case_count(tab.loc[anc, 'cases'])
        tab_display.loc[anc, 'case_pct_within_group'] = coarsen_case_pct(anc, tab.loc[anc, 'case_pct_within_group'])
        tab_display.loc[anc, 'pct_of_all_cases'] = coarsen_case_pct(anc, tab.loc[anc, 'pct_of_all_cases'])

print(tab_display.loc[tab.sort_values('cases', ascending=False).index].to_string())


## Save


In [ ]:
# Saving Table 1 and the cohort with baseline characteristics merged in
table1_df.to_csv(f"{bucket_output}/table1_baseline_characteristics.csv", index=False)
final_cohort.to_csv(f"{bucket_output}/03_ec_cohort_with_baseline_chars.csv", index=False)
print(f"Saved table1_baseline_characteristics.csv and 03_ec_cohort_with_baseline_chars.csv")
print(f"Cohort: N={len(final_cohort):,} "
      f"({int((final_cohort['EC_case']==1).sum()):,} cases, {int((final_cohort['EC_case']==0).sum()):,} controls)")
